In [7]:
import numpy as np
import time

# PHASE 1: Setup Parameters
np.random.seed(42)  # For reproducible results

# Portfolio settings
INITIAL_PORTFOLIO = 1_000_000
WEIGHTS = np.array([0.2, 0.2, 0.2, 0.2, 0.2])  # Equal weight (20% each)
N_ASSETS = len(WEIGHTS)
N_DAYS = 252           # Trading days in a year
N_SIMULATIONS = 50_000 # Number of possible futures

# Annual expected returns and volatilities (example values)
annual_returns = np.array([0.08, 0.12, 0.06, 0.10, 0.05])   # 8%, 12%, etc.
annual_volatility = np.array([0.20, 0.25, 0.15, 0.22, 0.18]) # 20%, 25%, etc.

# Convert to daily values
daily_returns = annual_returns / N_DAYS
daily_vol = annual_volatility / np.sqrt(N_DAYS)

# Correlation matrix (5x5)
corr_matrix = np.array([
    [1.00, 0.60, 0.30, 0.40, 0.20],
    [0.60, 1.00, 0.20, 0.50, 0.10],
    [0.30, 0.20, 1.00, 0.30, 0.50],
    [0.40, 0.50, 0.30, 1.00, 0.40],
    [0.20, 0.10, 0.50, 0.40, 1.00]
])

In [8]:
# PHASE 2: Cholesky Decomposition (Correlation Recipe)
L = np.linalg.cholesky(corr_matrix)  # Lower triangular mixing matrix

In [9]:
# PHASE 3-5: Monte Carlo Simulation Function
def run_simulation():
    """Run 50,000 portfolio simulations and return final values."""
    # Generate independent standard normal random numbers
    uncorrelated = np.random.randn(N_DAYS, N_SIMULATIONS, N_ASSETS)
    
    # Apply correlation via Cholesky factor
    correlated = uncorrelated @ L.T
    
    # Convert to daily returns: return = mean + volatility * shock
    daily_shocks = daily_vol * correlated + daily_returns
    
    # Cumulative growth: product of (1 + return) over all days
    cumulative_growth = np.cumprod(1 + daily_shocks, axis=0)
    
    # Final growth factor for each asset (last day)
    final_growth = cumulative_growth[-1]  # Shape: (50000, 5)
    
    # Initial capital per asset
    initial_per_asset = INITIAL_PORTFOLIO * WEIGHTS  # Shape: (5,)
    
    # Final portfolio values: sum across assets
    final_values = (initial_per_asset * final_growth).sum(axis=1)  # Shape: (50000,)
    
    return final_values

In [10]:
# PHASE 7: Run and Time
print("Running Monte Carlo simulation...")
start_time = time.perf_counter()
final_portfolio = run_simulation()
elapsed = time.perf_counter() - start_time


Running Monte Carlo simulation...


In [11]:
# PHASE 6: Risk Metrics Calculation
# Profit and Loss (P&L)
pnl = final_portfolio - INITIAL_PORTFOLIO

# Expected values
expected_final = np.mean(final_portfolio)
expected_return = (expected_final / INITIAL_PORTFOLIO) - 1

# Value at Risk (5th percentile of P&L)
var_95 = np.percentile(pnl, 5)

tail_mask = pnl <= var_95
cvar = pnl[tail_mask].mean()


In [13]:
# Output Results
print("\n" + "=" * 50)
print("MONTE CARLO PORTFOLIO RISK SIMULATION")
print("=" * 50)
print(f"Initial Investment:      ${INITIAL_PORTFOLIO:,.0f}")
print(f"Number of Simulations:   {N_SIMULATIONS:,}")
print(f"Time Horizon:            {N_DAYS} trading days (1 year)")
print("-" * 50)
print(f"Simulation Runtime:      {elapsed:.3f} seconds")
print("-" * 50)
print(f"Expected Final Value:    ${expected_final:,.2f}")
print(f"Expected Annual Return:  {expected_return * 100:.2f}%")
print("-" * 50)
print(f"5% Value at Risk (VaR):  ${-var_95:,.2f}")
print(f"   (5% chance of losing more than this amount)")
print(f"5% Conditional VaR:      ${-cvar:,.2f}")
print(f"   (Average loss in worst 5% of scenarios)")
print("=" * 50)

assert elapsed < 10.0, f"Simulation took {elapsed:.2f}s, should be <10s"
print("\nPerformance target met (under 10 second).")


MONTE CARLO PORTFOLIO RISK SIMULATION
Initial Investment:      $1,000,000
Number of Simulations:   50,000
Time Horizon:            252 trading days (1 year)
--------------------------------------------------
Simulation Runtime:      4.223 seconds
--------------------------------------------------
Expected Final Value:    $1,084,783.81
Expected Annual Return:  8.48%
--------------------------------------------------
5% Value at Risk (VaR):  $147,880.47
   (5% chance of losing more than this amount)
5% Conditional VaR:      $194,526.41
   (Average loss in worst 5% of scenarios)

Performance target met (under 10 second).
